# 빅데이터 분석기사 실기 모의고사 5회차

## 작업형 3유형 · 통계 분석 — 로지스틱 회귀 + Odds Ratio (3문항 × 10점 = 30점)

**출제 범위**: 단순 로지스틱 회귀 / 다중 로지스틱 회귀 + OR 비교 / 모델 적합도 (Pseudo R² + LLR 검정)
**사용 라이브러리**: `statsmodels.api` (sm.Logit) — sklearn 아님
**유의수준**: α = 0.05

> ⚠ **statsmodels 핵심**: `sm.add_constant(X)` 로 절편(intercept) 추가 필수.
> 본 회차는 **3·4회차의 OLS(연속형 회귀)** 학습 후 진도이며, **분류용 회귀 — 로지스틱 회귀** 주제(OR 해석·Wald z-검정·Pseudo R²) 를 다룹니다.

---

## 데이터 로드 (작업형 3 전용 — 의료 관행 target 재코딩)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

# 5회차 공통 데이터 — sklearn 내장 Breast Cancer (569명 × 31)
bc = load_breast_cancer(as_frame=True).frame
bc.columns = [c.replace(' ', '_') for c in bc.columns]

# 작업형 3 전용 — 의료 관행 target 재코딩
# sklearn 기본: 0=악성, 1=양성  →  의료 관행: 1=악성(질병), 0=양성
y = 1 - bc['target']            # 1 = malignant (the disease)
print(y.value_counts())
# 0 (benign):    357
# 1 (malignant): 212

---

## 문제 5-3-1 (10점) — 단순 로지스틱 회귀: mean_radius → 악성

`mean_radius` 를 독립변수, **악성 여부 (y=1=악성, 0=양성)** 를 종속변수로 하는
**단순 로지스틱 회귀모형** 을 적합하고, 회귀계수의 유의성을 검정하고 **Odds Ratio** 를 해석하시오.

### 모형
$$ \text{logit}(P) = \log\!\left(\frac{P}{1-P}\right) = \beta_0 + \beta_1 \cdot \text{mean\_radius} $$

(P = 악성일 확률, Odds = P/(1-P))

### 가설
- H₀: β₁ = 0 (mean_radius 는 악성 오즈에 영향 없음)
- H₁: β₁ ≠ 0 (mean_radius 는 악성 오즈에 영향 있음)

### 수행
`sm.Logit(y, X).fit()` → `sm.add_constant()` 필수 → `np.exp(β)` 로 OR 계산

### 출력 형식
1. **회귀계수 β₁** (소수점 둘째 자리)
2. **Odds Ratio = exp(β₁)** (소수점 둘째 자리)
3. **검정 결론**: "기각" 또는 "채택"

> 💡 **OR 해석 가이드**
> - OR > 1: 위험요인 (변수 1단위 증가 시 악성 오즈가 OR배 증가)
> - OR < 1: 보호요인
> - OR = 1: 영향 없음

In [ ]:
# 여기에 코드를 작성하세요.

# import statsmodels.api as sm
# X = sm.add_constant(bc['mean_radius'])
# model = sm.Logit(y, X).fit(disp=0)
# beta1 = model.params['mean_radius']
# or_value = np.exp(beta1)
# p_value  = model.pvalues['mean_radius']
# print(f'beta_1 = {beta1:.2f}')
# print(f'OR     = {or_value:.2f}')
# print('기각' if p_value < 0.05 else '채택')


---

## 문제 5-3-2 (10점) — 다중 로지스틱 회귀: Odds Ratio 비교

`mean_radius` 와 `mean_texture` **두 변수** 를 동시에 독립변수로 하는 **다중 로지스틱 회귀모형** 을 적합하시오.
각 변수의 **Odds Ratio** 와 **95% 신뢰구간** 을 계산하고, 어떤 변수가 더 강력한 위험요인인지 비교하시오.

### 모형
$$ \text{logit}(P_{악성}) = \beta_0 + \beta_1 \cdot \text{mean\_radius} + \beta_2 \cdot \text{mean\_texture} $$

### 수행
`sm.Logit` 적합 → `params` 로 회귀계수 → `np.exp(coef)` 로 OR → `model.conf_int()` 로 회귀계수 신뢰구간 → `np.exp()` 로 OR 신뢰구간 변환

### 출력 형식
1. **mean_radius 의 Odds Ratio** (소수점 둘째 자리)
2. **mean_texture 의 Odds Ratio** (소수점 둘째 자리)
3. **결론**: 두 변수의 OR 해석 (예: "두 변수 모두 양의 위험요인")

> 💡 **힌트**: OR 의 95% 신뢰구간이 **1을 포함하지 않으면** 통계적으로 유의함.
> 두 변수의 OR 신뢰구간이 **겹치지 않으면** 어느 쪽이 더 강력한 위험요인인지 통계적 결론 가능.

In [ ]:
# 여기에 코드를 작성하세요.

# X = sm.add_constant(bc[['mean_radius', 'mean_texture']])
# model = sm.Logit(y, X).fit(disp=0)
# for var in ['mean_radius', 'mean_texture']:
#     coef = model.params[var]
#     or_v = np.exp(coef)
#     ci   = model.conf_int().loc[var]
#     print(f'{var}: OR={or_v:.2f}, 95% CI=({np.exp(ci[0]):.2f}, {np.exp(ci[1]):.2f})')


---

## 문제 5-3-3 (10점) — 로지스틱 회귀 모델 적합도: Pseudo R² + LLR 검정

[5-3-2] 에서 적합한 다중 로지스틱 회귀 모델(`mean_radius` + `mean_texture` → 악성) 의
**전체 적합도** 를 평가하시오.

### 가설 (LLR 검정 — Likelihood Ratio Test)
- H₀: 모든 회귀계수 = 0 (모델이 무의미)
- H₁: 적어도 하나의 β ≠ 0 (모델이 유의미)

### 수행
`model.prsquared` (McFadden Pseudo R²), `model.llr_pvalue` (LLR 검정 p-value), `model.aic` (AIC) 활용

### 출력 형식
1. **Pseudo R²** (McFadden, 소수점 넷째 자리)
2. **AIC** (소수점 둘째 자리)
3. **검정 결론**: "기각" 또는 "채택" (LLR p-value 기준)

> 💡 **McFadden Pseudo R² 해석 가이드**
> - 0.0 ~ 0.2: 약한 적합
> - 0.2 ~ 0.4: 양호한 적합 (실무 일반적 수준)
> - 0.4 이상: 매우 우수한 적합
> (선형회귀 R² 보다 절댓값이 작게 나오므로 0.4 이상이면 거의 완벽)

In [ ]:
# 여기에 코드를 작성하세요.

# pseudo_r2 = model.prsquared
# llr_pval  = model.llr_pvalue
# aic       = model.aic
# print(f'Pseudo R^2 = {pseudo_r2:.4f}')
# print(f'AIC        = {aic:.2f}')
# print('기각' if llr_pval < 0.05 else '채택')


---

## 풀이 종료 안내

- **3유형 합격선**: 30점 만점 기준 18점 이상

### 로지스틱 회귀 vs OLS 핵심 차이 (5회차 핵심)
| 항목 | OLS (3·4회) | Logit (5회) |
|---|---|---|
| 종속변수 y | 연속형 | 이진 (0/1) |
| API | `sm.OLS(y, X)` | `sm.Logit(y, X)` |
| 검정통계량 | t-통계량 | **z-통계량 (Wald)** |
| 결정계수 | R² (`rsquared`) | **Pseudo R² (`prsquared`)** |
| 모델 전체 검정 | F-검정 (`fvalue`) | **LLR 검정 (`llr_pvalue`)** |
| 회귀계수 해석 | 단위 변화량 | **Odds Ratio = exp(β)** |

### 의료통계 OR 보고 표준 양식
`OR (95% CI: lo-hi, p < 0.001)` — 의학 논문/임상 보고서 표준

예시: "mean_radius 1단위 증가 시 악성 위험이 2.88배 증가했다 (95% CI: 2.36-3.51, p < 0.001)."

---

*폐쇄형 시험 환경 적응 모의고사 - 빅데이터 분석기사 실기 (sklearn Breast Cancer 데이터 기준)*